# Team and Project Meta Information

### Phase Leader

Daniel Yi (daniel_yi@berkeley.edu)

### Project Title

Turning Airline Delay Prediction into Operational Efficiency

### Group Number

Team 3_2

### Team Members

| Photo                                                                                                                                | Name        | Email                           |
|--------------------------------------------------------------------------------------------------------------------------------------|-------------|---------------------------------|
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/team_photos/daniel.png" width="100">  | Daniel Yi   | daniel_yi@berkeley.edu          |
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/team_photos/frank.png" width="100">   | Frank Lin   | frank_tingchun_lin@berkeley.edu |
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/team_photos/jason.png" width="100">   | Jason Chang | chan572@berkeley.edu            |
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/team_photos/william.png" width="100"> | William Shu | william.shu@berkeley.edu        |
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/team_photos/allen.png" width="100">   | Allen Li    | allen_li@berkeley.edu           |

### Phase Leader Plan

| **Project Phase** | **Leader(s)**             | **Email**                                        |
|-------------------|---------------------------|--------------------------------------------------|
| Week 1 (Phase 1)  | Frank Lin                 | frank_tingchun_lin@berkeley.edu                  |
| Week 2 (Phase 2)  | William Shu               | william.shu@berkeley.edu                         |
| Week 3 (Phase 2)  | Daniel Yi                 | daniel_yi@berkeley.edu                           |
| Week 4 (Phase 3)  | Allen Li                  | allen_li@berkeley.edu                            |
| Week 5 (Phase 3)  | Jason Chang               | chan572@berkeley.edu                             |

# Project Abstract

# Project Description

#### Task
We frame flight delay predictions as a binary classification task to determine two hours before scheduled departure whether a flight will experience a departure delay of at least 15 minutes or greater. Specifically, our labels will be:
1. **Delay** (≥15 min)
2. **No Delay** (< 15 min)

Furthermore, our primary clients will be the airlines, which rely on accurate early predictions to make informed operational decisions that directly affect efficiency and customer satisfaction.

#### Business Use Case

For airlines, departure delays create operational challenges that can snowball on top of each other. When a single flight runs late, crew schedules are disrupted, aircraft turnaround slows, gate assignments shift, and passengers risk missing connections. These disruptions reduce efficiency, erode customer satisfaction, and cost airlines millions of dollars each year.

However, reliable delay predictions delivered two hours before departure allow airlines to proactively adjust crew assignments, allocate gates more efficiently, and better coordinate baggage and transfer operations. By anticipating delays before they escalate, airlines can minimize operational ripple effects, improve resource planning, reduce financial losses, and provide a more seamless experience for passengers.

#### Dataset Description
To carry out our flight delay predictions, we use the On‑Time Flight Performance and Weather Dataset (OTPW). This dataset spans from 2015-2021 and combines hourly weather observations from the National Oceanic and Atmospheric Administration (NOAA), airport metadata from the Department of Transportation, and on‑time performance records for US flights from the Bureau of Transportation Statistics. We use a 12-month subset of this data from January 1 to December 31, 2015 which has dimensions `11623708 rows x 216 columns`. However, after deduplication, the dimensions reduce to `5811854 rows x 216 columns`. Each row represents a single flight paired with its origin and destination metadata, the corresponding hourly weather conditions, and the flight's recorded departure delay. The detailed data dictionary for this table is introduced in the following EDA section.

# EDA

### Feature Engineering

####  Missing data for Response Variable

As mentioned in the EDA, the variable we are interested in is if a flight is delayed over 15 minutes. Looking at reponse variable, it has 3.02 percent missing data, all caused by cancelled flights. To resolve this issue, We decide to remove all cancelled flights. This results in 5,722,067 rows remaining for the one-year OTPW dataset, with no more missing values in the delay response variable.

#### Other Missing data Explaination

A significant amount of missing data comes from the weather categories. However, we would like to use this data since we believe weather is an important predictor to weather or not a flight is delayed. Specifically, we are concerned with hourly weather categories.

One of the reasons why there are so many missing values is that the weather dataset has multiple report types (FM-15, FM-16, FM-12, etc) that do not contain all information. For example:

* FM-15: Standard hourly temperature, wind, visibility, pressure, and precipitation observations. This is the most complete and consistent report type, making it ideal for linking to individual flights.
* FM-16: Special weather report issued when conditions change significantly (e.g., sudden storms or extreme winds). It is irregular and event-driven, so coverage is sparse.
* FM-12: Synoptic report issued every 6–12 hours with fewer variables; many fields are missing and it does not align well with hourly flight data.
* SOD (Summary of Day): Daily aggregates like total precipitation or max/min temperature; not useful for per-flight modeling.
* SHEF and SY-MT: Rare or specialized reports (hydrology or system metadata) with minimal rows and missing data.

In the one-year OTPW dataset, the have the following distribution of report types:

| Report Type | count   |
|-------------|---------|
| SHEF        | 999     |
| FM-15       | 5005738 |
| SOD         | 11952   |
| SY-MT       | 2504    |
| FM-12       | 228622  |
| FM-16       | 472252  |

Because of this, we focus primarily on FM-15 for modeling. In the future we can optionally include FM-16 if we want to capture extreme weather events. As for the other report tyes, we will and drop them to ensure consistent and reliable features for each flight.

After filtering for only FM-15 report types, there are 5,005,738 rows remaining.

#### Normalization and Transformations

Based off of our EDA, we can see that precipitation, humidity, wind speed, visibility and temperature are the strongest flight delay signals. We will use them in our modeling, and they will need to be normalized and transformed.

| Before | After |
|--------|-------|
| <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/feature_eng/precip.png" width="500"><br> | <img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/feature_eng/precip_binary.png" width="468"><br> |

Precipitation is heavily skewed, even after log transformations it is still skewed. A better option would be to turn it into a binary feature where 0 = no precipitation and 1 = precipitation.

Next, we look at the other numeric features: humidity, wind speed, visibility and temperature. Relative humidity and temperature seem to be relatively normal. We normalize them so for use in ML modeling.

<img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/feature_eng/humidity_temp_normalized.png" width="900"><br>

Wind speed is right skewed. We will log transform it and then scale it.

<img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/feature_eng/wind_log_normalized.png" width="500"><br>

Visibility is sparce and left skewed, with a max. We decide to make it into a categorical variable instead.

| Category | Description                      |
| -------- | -------------------------------- |
| 0        | Very Low Visibility (< 2 miles)  |
| 1        | Low Visibility (2–5 miles)       |
| 2        | Moderate Visibility (5–10 miles) |
| 3        | High Visibility (>= 10 miles)    |

<br><img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/feature_eng/visibility_cat.png" width="500"><br>

#### Feature Creation

Alongside the existing features, we will create two additional features to capture temporal and route-specific patterns in flight delays. First, we introduce a binary indicator for whether a flight occurs on a weekend (Saturday or Sunday). Travel patterns and airport congestion often differ between weekdays and weekends, which can influence delay behavior, making this a useful feature for the model.

Second, we construct a combined feature representing the route by concatenating the origin and destination airports. This allows the model to capture route-specific effects, as certain routes may consistently experience higher delays due to factors such as traffic volume, weather patterns, or airport efficiency. By encoding this combined route variable, we preserve important interaction information between origin and destination that would be lost if they were treated independently.

# Modeling Pipelines

# Results and Discussion of Results

# Conclusion

# Updated Credit assignment plan

### Phase 1 (Due 3/15)

#### Task List

| **Task**                                 | **Task Type** | **Effort Hours** | **Assigned To**          |
|------------------------------------------|---------------|------------------|--------------------------|
| Credit Assignment Plan                   | Report        |                2 | Daniel Yi                |
| Project Abstract                         | Report        |                2 | Frank Lin                |
| Data Description (1)                     | Report        |                1 | Frank Lin                |
| Data Description (2)                     | Report        |                1 | William Shu              |
| EDA for Weather Dataset                  | EDA           |                4 | Allen Y Li               |
| EDA for Airports + Airport Codes Dataset | EDA           |                2 | Frank Lin                |
| EDA for Flights Dataset                  | EDA           |                2 | Daniel Yi                |
| EDA for OTPW Dataset (1)                 | EDA           |                2 | William Shu              |
| EDA for OTPW Dataset (2)                 | EDA           |                3 | Jason Chang              |
| Summary EDA for All Datasets (1)         | Report        |                1 | Frank Lin                |
| Summary EDA for All Datasets (2)         | Report        |                1 | Jason Chang              |
| Machine Learning Algorithm Discussion (1)| Modeling      |                2 | Allen Y Li               |
| Machine Learning Algorithm Discussion (2)| Modeling      |                2 | Frank Lin                |
| Metrics Discussion (1)                   | Report        |                1 | Frank Lin                |
| Metrics Discussion (2)                   | Report        |                1 | William Shu              |
| Gantt Chart                              | Report        |                1 | Daniel Yi                |
| Pipeline Description (1)                 | Modeling      |                1 | Jason Chang              |
| Pipeline Description (2)                 | Modeling      |              0.5 | William Shu              |
| Team Member Emails                       | Report        |              0.5 | Daniel Yi                |
| Feature Engineering Plan (1)             | Report        |              0.5 | Daniel Yi                |
| Feature Engineering Plan (2)             | Report        |              0.5 | William Shu              |
| Open Problems + Next Steps (1)           | Report        |                1 | Frank Lin                |
| Open Problems + Next Steps (2)           | Report        |                1 | William Shu              |
| Report Finalization                      | Report        |                2 | Frank Lin                |

#### Cumulative Hours

| Name        | Total Hours |
|-------------|-------------|
| Daniel Yi   | 7           |
| Frank Lin   | 12          |
| Jason Chang | 5           |
| William Shu | 6           |
| Allen Li    | 6           |


### Phase 2 (Due 4/5)

#### Task List

| Task                                                                                                                       | Task Type         | Effort Hours | Assigned To |
|----------------------------------------------------------------------------------------------------------------------------|-------------------|--------------|-------------|
| EDA for Weather Dataset (Cont.)                                                                                            | EDA               |            0 | Allen Y Li  |
| EDA for Airports + Airport Codes Dataset (Cont.)                                                                           | EDA               |            0 | Frank Lin   |
| EDA for Flights Dataset (Cont.)                                                                                            | EDA               |            0 | Jason Chang |
| EDA for OTPW Dataset (Cont.)                                                                                               | EDA               |            0 | William Shu |
| Joining Data (Extra Credit)                                                                                                | Modeling          |            0 | William Shu |
| Feature Engineering + Feature Transformation + Discussion                                                                  | Report + Modeling |            0 | Daniel Yi   |
| Model Three-month OTPW Dataset and perform experiments                                                                     | Modeling          |            0 | William Shu |
| Model One-year OTPW Dataset and perform experiments                                                                        | Modeling          |            0 | William Shu |
| Model Experiment Discussion + Cross Validation                                                                             | Report            |            0 | William Shu |
| Create new features that are highly predictive: at least one time-based feature, e.g., recency, frequency, monetary, (RFM) | Report + Modeling |            0 | Daniel Yi   |
| Team and project meta information.                                                                                         | Report            |            0 | Daniel Yi   |
| Discuss Modeling piplines and features                                                                                     | Report            |            0 | Jason Chang |
| Updated Credit Assignment Plan                                                                                             | Report            |            0 | Daniel Yi   |
| Updated Abstract                                                                                                           | Report            |            0 | Frank Lin   |
| EDA                                                                                                                        | Report            |            0 | Jason Chang |
| Project Description                                                                                                        | Report            |            0 | Allen Y Li  |
| Updated Conclusion + Next Steps                                                                                            | Report            |            0 | Frank Lin   |
| In-class Presentation (1)                                                                                                  | Presentation      |            0 | Allen Y Li  |
| In-class Presentation (2)                                                                                                  | Presentation      |            0 | Frank Lin   |
| In-class Presentation (3)                                                                                                  | Presentation      |            0 | William Shu |
| In-class Presentation (4)                                                                                                  | Presentation      |            0 | Daniel Yi   |
| In-class Presentation (5)                                                                                                  | Presentation      |            0 | Jason Chang |

#### Cumulative Hours

| Name        | Total Hours |
|-------------|-------------|
| Daniel Yi   |           0 |
| Frank Lin   |           0 |
| Jason Chang |           0 |
| William Shu |           0 |
| Allen Li    |           0 |


### Phase 3 (Due 4/5)

#### Task List

| **Task**                                                                                | **Task Type**     | **Effort Hours** | **Assigned To** |
|-----------------------------------------------------------------------------------------|-------------------|------------------|-----------------|
| Consider more sophisticated models (GBDT, MLP neural networks, etc.)                    | Report + Modeling |                  | TBD             |
| Fine-tune your pipeline using a grid search, or any search techniques                   | Report + Modeling |                  | TBD             |
| Do experiments on ALL the data for new features and experimental settings               | Report + Modeling |                  | TBD             |
| Feature engineering and refinement (at least one Time-based feature and Graph feature)  | Report + Modeling |                  | TBD             |
| Exciting novel directions that you pursued                                              | Report + Modeling |                  | TBD             |
| Final Abstract                                                                          | Report            |                  | TBD             |
| Data and Feature Engineering (EDA)                                                      | Report            |                  | TBD             |
| Neural Network (MLP)                                                                    | Report            |                  | TBD             |
| Leakage                                                                                 | Report            |                  | TBD             |
| Modeling Pipeline                                                                       | Report            |                  | TBD             |
| Results and discussion of results + Gap Analysis of Best Model                          | Report            |                  | TBD             |
| Concusion                                                                               | Report            |                  | TBD             |
| Clean Code + Report Finalization                                                        | Report            |                  | TBD             |
| In-class Presentation                                                                   | Presentation      |                  | TBD             |

#### Cumulative Hours

| Name        | Total Hours |
|-------------|-------------|
| Daniel Yi   |           0 |
| Frank Lin   |           0 |
| Jason Chang |           0 |
| William Shu |           0 |
| Allen Li    |           0 |

### Gantt 

<img src="https://raw.githubusercontent.com/william-shu-ucb/261-Final-Team-3_2/refs/heads/main/images_phase_2/Project Task List - Gantt Chart-1.png" width="900"><br>